# Appendix Notebook A2: Asian Option Pricing with Hybrid Quantum Amplitude Estimation

This appendix notebook is the standardised review-paper version of the original local working file for Module 2.

This notebook implements the executable pricing workflow referenced in Module 2. It combines classical Monte Carlo simulation, histogram discretisation, amplitude encoding, and a PennyLane-based QAE-style estimation pipeline.

## Reading Guide

The notebook is organised around market setup, path simulation, payoff normalisation, circuit construction, ancilla measurement, and cross-checking against classical pricing outputs.

All file names in `review_paper/notebooks/` follow a common naming convention so the paper appendix can reference them consistently.

## Imports and Environment

This section loads the Python packages required for the pricing experiment and sets the numerical environment for reproducibility.

In [ ]:
# =========================
# PennyLane QAE for Asian option (distribution-matched)
# =========================
import pennylane as qml
import numpy as np

## Market and Simulation Settings

The financial contract, market parameters, and Monte Carlo controls are defined here.

In [ ]:
# ---------- 1) Market & simulation settings ----------
S0, K = 100.0, 100.0
r, sigma, T = 0.02, 0.20, 1.0
n_steps = 32              # steps for Asian average
n_paths = 200_000         # MC paths; same data will feed QAE histogram
rng = np.random.default_rng(123)

## Simulated Path Construction

Geometric Brownian motion paths are simulated and converted into arithmetic Asian averages.

In [ ]:
# ---------- 2) Simulate GBM paths & Asian averages ----------
dt = T / n_steps
Z = rng.standard_normal((n_paths, n_steps))
logS = np.log(S0) + np.cumsum((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z, axis=1)
S_paths = np.exp(logS)
S_bar = S_paths.mean(axis=1)

# Classical MC price (as ground truth under the SAME law)
disc = np.exp(-r*T)
V_MC = disc * np.maximum(S_bar - K, 0.0).mean()

## Histogram Encoding

The simulated payoff distribution is discretised into histogram bins so it can be mapped to a quantum amplitude-estimation workflow.

In [ ]:
# ---------- 3) Build histogram p_i for QAE (same law as MC) ----------
nP = 7                    # number of price-register qubits  -> bins = 2^nP
bins = 2**nP
mu, sd = S_bar.mean(), S_bar.std()
Smin, Smax = max(1e-8, mu - 4*sd), mu + 4*sd
edges = np.linspace(Smin, Smax, bins+1)
centers = 0.5*(edges[:-1] + edges[1:])
counts, _ = np.histogram(S_bar, bins=edges)
p = counts.astype(float) / max(1, counts.sum())            # safe normalize
sqrt_p = np.sqrt(p + 0.0)                                  # amplitudes

## Payoff Normalisation

The payoff is rescaled to the unit interval so the ancilla amplitude can encode the target expectation safely.

In [ ]:
# ---------- 4) Scaled payoff f' in [0,1], choose B consistent with histogram range ----------
B = max(centers.max() - K, 1e-12)                          # upper bound
fprime = np.clip((centers - K)/B, 0.0, 1.0)
thetas = 2.0*np.arcsin(np.sqrt(fprime))                    # angle per bin

## Device Configuration

Separate exact and shot-based PennyLane devices are configured to distinguish idealised circuit validation from finite-sampling estimation.

In [ ]:
# ---------- 5) PennyLane device(s) ----------
price_wires = list(range(nP))
anc_wire = nP
all_wires = price_wires + [anc_wire]

# Analytic (statevector) device for exact ancilla probability
dev_exact = qml.device("default.qubit", wires=all_wires, shots=None)

# Optional sampling device (set shots>0 to see sampling noise)
shots = 50_000
dev_sample = qml.device("default.qubit", wires=all_wires, shots=shots)

## Controlled-Rotation Logic

This section defines the pattern-controlled rotations that embed the discretised payoff into the ancilla qubit.

In [ ]:
# ---------- 6) Robust multi-control on arbitrary bit pattern ----------
def apply_pattern_controlled_RY(theta, ctrl_wires, bits, anc_wire):
    """
    Apply RY(theta) on ancilla conditioned on ctrl_wires matching 'bits' (0/1 list).
    Implementation: X-sandwich to map the pattern to all-ones, then use all-ones
    multi-controlled RY, then uncompute the X's.
    """
    # 1) Flip 0-bits to 1 by X
    for w, b in zip(ctrl_wires, bits):
        if b == 0:
            qml.PauliX(w)
    # 2) All-ones multi-controlled RY
    CRY_all1 = qml.ctrl(qml.RY, control=ctrl_wires)  # control=all 1's
    CRY_all1(theta, wires=anc_wire)
    # 3) Unflip
    for w, b in zip(ctrl_wires, bits):
        if b == 0:
            qml.PauliX(w)

def apply_controlled_ancilla_rotations(thetas, price_wires, anc_wire, order="MSB"):
    """
    Dispatch pattern-controlled RY for each basis index i using a chosen bit order.

    order="MSB": bits are MSB->LSB, ctrl_wires = price_wires as given.
    order="LSB": bits are LSB->MSB, ctrl_wires = reversed(price_wires).

    This covers both common conventions; we'll auto-pick which matches  p_i f'_i.
    """
    nP = len(price_wires)
    if order.upper() == "MSB":
        ctrl_wires = list(price_wires)               # MSB ... LSB
        for i, theta in enumerate(thetas):
            bits_msb2lsb = [(i >> b) & 1 for b in reversed(range(nP))]  # MSBLSB
            apply_pattern_controlled_RY(theta, ctrl_wires, bits_msb2lsb, anc_wire)
    elif order.upper() == "LSB":
        ctrl_wires = list(reversed(price_wires))     # treat first as LSB
        for i, theta in enumerate(thetas):
            bits_lsb2msb = [(i >> b) & 1 for b in range(nP)]            # LSBMSB
            apply_pattern_controlled_RY(theta, ctrl_wires, bits_lsb2msb, anc_wire)
    else:
        raise ValueError("order must be 'MSB' or 'LSB'")


## QNode Definitions

The QNodes implement the exact-probability and sampled versions of the pricing circuit.

In [ ]:
# ---------- 7) QNodes (unchanged except an 'order' arg) ----------
@qml.qnode(dev_exact)
def ancilla_prob_exact(sqrt_p, thetas, order):
    qml.AmplitudeEmbedding(sqrt_p, wires=price_wires, normalize=False)
    apply_controlled_ancilla_rotations(thetas, price_wires, anc_wire, order=order)
    return qml.probs(wires=anc_wire)

@qml.qnode(dev_sample)
def ancilla_prob_sample(sqrt_p, thetas, order):
    qml.AmplitudeEmbedding(sqrt_p, wires=price_wires, normalize=False)
    apply_controlled_ancilla_rotations(thetas, price_wires, anc_wire, order=order)
    return qml.probs(wires=anc_wire)

## Execution and Benchmarking

The circuit outputs are compared with the classical Monte Carlo benchmark to quantify consistency.

In [ ]:
# ---------- 8) Run and compare ----------
a_hist = float((p * fprime).sum())  # ground-truth target from histogram
probs_M = ancilla_prob_exact(sqrt_p, thetas, order="MSB"); a_M = float(probs_M[1])
probs_L = ancilla_prob_exact(sqrt_p, thetas, order="LSB"); a_L = float(probs_L[1])

err_M = abs(a_M - a_hist)
err_L = abs(a_L - a_hist)
chosen = "MSB" if err_M <= err_L else "LSB"
a_exact = a_M if chosen == "MSB" else a_L
V_QAE_exact = disc * B * a_exact

# Sampled estimate under chosen order
probs_samp = ancilla_prob_sample(sqrt_p, thetas, order=chosen)
a_samp = float(probs_samp[1]); V_QAE_sample = disc * B * a_samp

print("=== PennyLane QAE (endianness auto-calibrated; X-sandwich controls) ===")
print(f" Discount factor e^(-rT) = {disc:.6f},  Histogram range [{Smin:.4f}, {Smax:.4f}], bins={bins}")
print(f" B (from range) = {B:.6f}")
print(f" MC (direct)      : {V_MC:.6f}    (target a* = {V_MC/(disc*B):.6f})")
print(f" a_hist ( p_i f') : {a_hist:.6f}")
print(f" a_MSB={a_M:.6f} (||={err_M:.2e}), a_LSB={a_L:.6f} (||={err_L:.2e})  -> chosen: {chosen}")
print(f" QAE exact         : a = {a_exact:.6f} -> price {V_QAE_exact:.6f}")
print(f" QAE shots         : a = {a_samp:.6f} -> price {V_QAE_sample:.6f} (shots={shots})")

## Optional Grover Sanity Check

A short verification step illustrates the amplification and de-amplification logic behind the broader QAE intuition.

In [ ]:
# ---------- 9) (Optional) one-step Grover amplification demonstration ----------
# If you want to show m=1 and proper de-amplification on the *same* a_exact:
m = 1
theta0 = np.arcsin(np.sqrt(a_exact))
a1 = np.sin((2*m+1)*theta0)**2
theta_hat = np.arcsin(np.sqrt(a1)) / (2*m+1)
a_hat = np.sin(theta_hat)**2
print("\n--- Grover m=1 (deamp) check ---")
print(f" a_exact={a_exact:.6f} -> a1={a1:.6f} -> a_hat={a_hat:.6f}")

## Interpretation Note

The executable implementation in this notebook underpins the pricing discussion in Module 2, while the main text now keeps the narrative focus on method, assumptions, and interpretation rather than inline code.